In [19]:
/*
  MicroGrad by Andrej Karpathy
  https://www.youtube.com/watch?v=VMj-3S1tku0
  
  Translated to JavaScript by Joshua Moore
*/

// A Value encapsulates a number, as well as the operators and inputs 
// that led to it.
let Value = class Value {
  constructor(data, children=[], op='', label=''){
    this.data = data;
    this.prev = new Set(children);
    this.op = op;
    this.label = label;
    this.grad = 0;
    this._backward = () => {};
  }

  toString(){
    return `Value ${this.label + ' '}(data=${this.data.toFixed(3)} grad=${this.grad.toFixed(3)})`;
  }

  add(other){
    other = other instanceof Value ? other : new Value(other);
    const out = new Value(this.data + other.data, [this, other], '+');
    out._backward = () => {
      this.grad += out.grad;
      other.grad += out.grad;
    };
    
    return out;
  }

  mul(other){
    other = other instanceof Value ? other : new Value(other)
    const out = new Value(this.data * other.data, [this, other], '*');
    out._backward = () => {
      this.grad += other.data * out.grad;
      other.grad += this.data * out.grad;
    }
    
    return out;
  }

  pow(other){
    console.assert(`Argument other must be a Number.`, other instanceof Number);
    const out = new Value(this.data**other, [this], `**${other}`);
    out._backward = () => {
      this.grad += (other * this.data**(other-1)) * out.grad;
    }

    return out;
  }

  relu(){
    const out = new Value(this.data < 0 ? 0 : this.data, [this], 'ReLu');
    out._backward = () => {
      this.grad += (out.data > 0 ? 1 : 0) * out.grad;
    }

    return out;
  }

  backward(){
    const topo = [];
    const visited = new Set();
    const buildTopo = (v) => {
      if(!visited.has(v)){
        visited.add(v);
        for(let child of v.prev){
          buildTopo(child);
        }
        topo.push(v);
      }
    };
    buildTopo(this);

    this.grad = 1.0;
    for(let i = topo.length -1; i>=0; i--){
      topo[i]._backward();
    }
    
    return topo;
  }

  neg(){
    return this.mul(-1);
  }

  sub(other){
    other = other instanceof Value ? other : new Value(other);
    return this.add(other.neg());
  }

  div(other){
    return this.mul(other.pow(-1));
  }

  tanh(){
    const x = this.data;
    const t = (Math.exp(2*x) - 1)/(Math.exp(2*x) + 1);
    const out = new Value(t, [this], 'tanh');
    out._backward = () => {
      this.grad += (1- t**2) * out.grad;
    }

    return out;
  }

  exp(){
    const x = this.data;
    const out = new Value(Math.exp(x), [this], 'exp');
    out._backward = () => {
      this.grad += out.data * out.grad;
    }

    return out;
  }

  trace(){
    const nodes = new Set(), edges = new Set();
    const build = (v) => {
      nodes.add(v);
      for(let child of v.prev){
        edges.add([child, v]);
        build(child);
      }
    }
    build(root);
    return nodes, edges;
  }
}

// A Neuron of Values
let Neuron = class Neuron {
  constructor(nin, nonlin=true){
    this.w = Array(nin).fill(0).map(() => new Value((Math.random() * 2) - 1));
    this.b = new Value(0);
    this.nonlin = nonlin;
  }

  call(x){
    const act = this.w
      .filter((w, i) => i >= this.b.data)
      .map((wi, i) => [wi, x[i]])
      .reduce((acc, pair) => {
        return acc.add(pair[0].mul(pair[1]))
      }, new Value(0));
    
    const out = act.tanh();
    return out;
  }

  parameters(){
    return this.w.concat([this.b]);
  }

  toString(){
    return `${this.nonlin ? 'ReLu' : 'Linear'} Neuron(${this.w.length})`;
  }
}

// A Layer of Neurons
let Layer = class Layer {
  constructor(nin, nout, nonlin){
    this.neurons = Array(nout).fill(0).map(() => new Neuron(nin, nonlin));
  }

  call(x){
    const outs = this.neurons.map(n => n.call(x));
    return outs.length == 1 ? outs[0] : outs;
  }

  parameters(){
    const params = [];
    for(let neuron of this.neurons){
      for(let p of neuron.parameters()){
        params.push(p);
      }
    }
    
    return params;
  }

  toString(){
    return `Layer of [{${this.neurons.map(n => n.toString()).join(', ')}}]`
  }
}

// MultiLayerPerceptron
let MLP = class MLP {
  constructor(nin, nouts){
    const sz = [nin].concat(...nouts);
    this.layers = Array(nouts.length).fill(0).map((_, i) => new Layer(sz[i], sz[i+1], i < nouts.length - 1))
  }

  call(x){
    for(let layer of this.layers){
      x = layer.call(x);
    }
    
    return x;
  }

  parameters(){
    let params = [];
    for(let layer of this.layers){
      for(let p of layer.parameters()){
        params.push(p);
      }
    }

    return params;
  }

  toString(){
    return `MLP of [{${this.layers.map(l => l.toString()).join(', ')}}]`
  }
}

// Training
let gen = (from, to, step) => {
  const out = [];
  for(let x=from; x<=to; x+=step){
    out.push(x);
  }
  return out;
}

// define function we want to approximate
let fn = (x) => Math.sin(x);

// split data into train and test
//let xs = [gen(0, Math.PI, 0.1)];
//let ys = [gen(0, Math.PI, 0.1).map(fn)];

let xs = [
  [2, 3, -1],
  [3, -1, .5],
  [.5, 1, 1],
  [1, 1, -1]
];

let ys = [
  1, 
  -1, 
  -1, 
  1
];

console.log(`Xs: ${JSON.stringify(xs)}`);
console.log(`Ys: ${JSON.stringify(ys)}`)

let trainXs = [];
let trainYs = [];

let testXs = [];
let testYs = [];

for(let i=0; i<xs.length; i++){
  if(true){ //Math.random() < 0.75){
    trainXs.push(xs[i]);
    trainYs.push(ys[i]);
  }else{
    testXs.push(xs[i]);
    testYs.push(ys[i]);
  }
}

let mlp = new MLP(3, [4, 4, 1]);
console.log(`MLP(3, [4, 4, 1]), randomly initialized to [${[...mlp.parameters().map(p => p.data.toFixed(3))].join(", ")}]`);

const ROUNDS = 100, NABLA = 0.01;
let predYs = trainXs.map(x => mlp.call(x));
let loss = trainYs
 .map((ty, i) => predYs[i].sub(ty).pow(2))
 .reduce((sum, l) => sum.add(l), new Value(0));

// log values to output so we can see
console.log(`Initial Prediction: [${[...predYs.map(y => y.data.toFixed(3))].join(", ")}]`);
console.log(`Initial Loss: ${loss.data.toFixed(3)}`);

console.log(`${ROUNDS} rounds of training at a rate of ${NABLA}...`)
for(let i=0; i<ROUNDS; i++){
  // predict
  predYs = trainXs.map(x => mlp.call(x));
  loss = trainYs
    .map((ty, i) => predYs[i].sub(ty).pow(2))
    .reduce((sum, l) => sum.add(l), new Value(0));

  // reset gradients
  // mlp.parameters().forEach(p => p.grad = 0.0);
  let params = mlp.parameters();
  for(let p of params){
    p.grad = 0;
  }

  // calculate gradients
  loss.backward();

  // tweak parameters
  params = mlp.parameters();
  for(let p of params){
    p.data += -NABLA * p.grad;
  }
  
  // console.log(`Iteration: ${i} Loss: ${loss.data.toFixed(2)}`)
}
console.log(`... done`)

console.log(`Trained Prediction: [${[...predYs.map(y => y.data.toFixed(3))].join(", ")}]`);
console.log(`Trained Loss: ${loss.data.toFixed(3)}`);


Xs: [[2,3,-1],[3,-1,0.5],[0.5,1,1],[1,1,-1]]Ys: [1,-1,-1,1]MLP(3, [4, 4, 1]), randomly initialized to [-0.858, 0.345, 0.046, 0.000, -0.066, -0.838, 0.252, 0.000, 0.064, 0.319, -0.227, 0.000, 0.866, -0.812, -0.052, 0.000, -0.182, -0.806, -0.573, 0.964, 0.000, -0.822, -0.448, -0.901, 0.560, 0.000, 0.560, 0.323, -0.082, 0.951, 0.000, 0.137, 0.966, 0.939, -0.104, 0.000, -0.940, -0.939, 0.994, -0.305, 0.000]Initial Prediction: [-0.493, -0.722, -0.339, -0.812]Initial Loss: 6.027100 rounds of training at a rate of 0.01...... doneTrained Prediction: [0.940, -0.943, -0.912, 0.902]Trained Loss: 0.024

undefined